In [1]:
import sys, os, h5py, time, psutil

import numpy as np
import healpy as hp
import tensorflow as tf

from tqdm import tqdm, trange
from sklearn.neighbors import BallTree

from deepsphere.utils import split_sparse_dense_matmul

### constants

In [2]:
nside = 512
npix = hp.nside2npix(nside)

sigma_arcmin = 15
n_sigma_support = 3
sigma_rad = sigma_arcmin / 60 / 180 * np.pi

### example map

In [3]:
# get healpix map
m = np.ones(npix)
cl = hp.alm2cl(hp.map2alm(m))
cl = 1 - np.arange(len(cl)) * 0.002
m = hp.synfast(cl, nside=nside, pixwin=True)
m = m_full = m - np.mean(m)
m = np.float32(m)
print(f"nside = {nside}, npix = {npix}")

# smooth
m_smooth_healpy = hp.sphtfunc.smoothing(m, sigma=sigma_rad)

# reduce to ~8000 square degrees
lon, lat = hp.pix2ang(nside, ipix=np.arange(npix), lonlat=True)
theta = np.vstack([np.radians(lat), np.radians(lon)]).T
ind_base = hp.ang2pix(nside=1, theta=lon, phi=lat, lonlat=True)
select = ind_base < 2
theta = theta[select, :]
m = m[select]
m_smooth_healpy = m_smooth_healpy[select]
npix_full = npix
npix = len(m)

print(f"new npix = {npix}, new area = {npix/npix_full*44000}")

nside = 512, npix = 3145728
new npix = 524288, new area = 7333.333333333333


### tree

In [4]:
# print(f"creating tree for {len(theta)} pixels and radius {n_sigma_support*sigma_arcmin} arcmin")
# tree = BallTree(theta, metric="haversine")

# inds_r, dist_r = tree.query_radius(theta, r=sigma_rad * n_sigma_support, return_distance=True, sort_results=True)
# n_neighbours = [len(i) for i in inds_r]
# max_neighbours = np.max(n_neighbours)
# print(f"max_neighbours = {max_neighbours}")

# theta_split = np.array_split(theta, 100)
# list_dist_k, list_inds_k = [], []
# for theta_ in tqdm(theta_split):
#     dist_k, inds_k = tree.query(theta_, k=max_neighbours, return_distance=True, sort_results=True)
#     list_dist_k.append(dist_k)
#     list_inds_k.append(inds_k)
# dist_k = np.concatenate(list_dist_k, axis=0)
# inds_k = np.concatenate(list_inds_k, axis=0)
# print(inds_k.shape, dist_k.shape)

# np.save(f"dist_k_nside{nside}.npy", dist_k)
# np.save(f"inds_k_nside{nside}.npy", inds_k)

# print(f"stored dist_k = {dist_k.shape}, inds_k = {inds_k.shape}")

In [5]:
dist_k = np.load(f"dist_k_nside{nside}.npy")
inds_k = np.load(f"inds_k_nside{nside}.npy")
max_neighbours = inds_k.shape[1]

### kernel

In [6]:
kernel_func = lambda r: np.exp(-0.5 / sigma_rad**2 * r**2)

kernel = kernel_func(dist_k)
kernel = np.float32(kernel)
inds_k = np.int64(inds_k)

print(f"kernel size {kernel.nbytes/1e9:4.2f} GB dtype {kernel.dtype}")
print(f"index size {inds_k.nbytes/1e9:4.2f} GB dtype {inds_k.dtype} max_ind={np.max(inds_k)}")
print(f"single map tensor {npix * max_neighbours * 4/1e9:4.2f} GB")

kernel size 0.29 GB dtype float32
index size 0.59 GB dtype int64 max_ind=524287
single map tensor 0.29 GB


In [7]:
n_channels = 8
batch_size = 26

In [8]:
# %%time
# with tf.device("cpu"):
    
#     kernel_channels = []
#     map_channels_batch = []
#     for i in range(n_channels):
#         inds_r = tf.constant(np.arange(npix), dtype=tf.int64)
#         inds_r = tf.expand_dims(inds_r, axis=-1)
#         inds_r = tf.tile(inds_r, (1, max_neighbours))
#         inds_c = tf.constant(inds_k, dtype=tf.int64)
#         ind_coo = tf.concat([tf.reshape(inds_r, (-1, 1)), tf.reshape(inds_c, (-1, 1))], axis=-1)
#         val_kernel = tf.reshape(kernel, [-1])

#         m_batch = tf.concat([tf.expand_dims(m, axis=-1)] * batch_size, axis=-1)
#         map_channels_batch.append(m_batch)

#         sparse_kernel = tf.sparse.SparseTensor(indices=ind_coo, values=val_kernel, dense_shape=[npix, npix])
#         sparse_kernel = tf.sparse.reorder(sparse_kernel)
#         kernel_channels.append(sparse_kernel)

#         print(f"ind_coo.shape={ind_coo.shape} ind_coo.size={np.array(ind_coo).nbytes/1e9:2.4f} GB val_kernel.shape={val_kernel.shape} val_kernel.size={np.array(val_kernel).nbytes/1e9:2.4f} GB")

# print(f"created {n_channels} sparse kernels with shape {kernel_channels[0].shape} and batched maps with size {map_channels_batch[0].shape}")

In [9]:
%%time   
kernel_channels = []
map_channels_batch = []
for i in range(n_channels):
    inds_r = tf.constant(np.arange(npix), dtype=tf.int64)
    inds_r = tf.expand_dims(inds_r, axis=-1)
    inds_r = tf.tile(inds_r, (1, max_neighbours))
    inds_c = tf.constant(inds_k, dtype=tf.int64)
    ind_coo = tf.concat([tf.reshape(inds_r, (-1, 1)), tf.reshape(inds_c, (-1, 1))], axis=-1)
    val_kernel = tf.reshape(kernel, [-1])

    m_batch = tf.concat([tf.expand_dims(m, axis=-1)] * batch_size, axis=-1)
    map_channels_batch.append(m_batch)

    sparse_kernel = tf.sparse.SparseTensor(indices=ind_coo, values=val_kernel, dense_shape=[npix, npix])
    sparse_kernel = tf.sparse.reorder(sparse_kernel)
    kernel_channels.append(sparse_kernel)

    print(f"ind_coo.shape={ind_coo.shape} ind_coo.size={np.array(ind_coo).nbytes/1e9:2.4f} GB val_kernel.shape={val_kernel.shape} val_kernel.size={np.array(val_kernel).nbytes/1e9:2.4f} GB")

print(f"created {n_channels} sparse kernels with shape {kernel_channels[0].shape} and batched maps with size {map_channels_batch[0].shape}")

ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
ind_coo.shape=(73400320, 2) ind_coo.size=1.1744 GB val_kernel.shape=(73400320,) val_kernel.size=0.2936 GB
created 8 sparse kernels with shape (524288, 524288) and batched maps with size (524288, 26)
CPU times: user 3.74 s, sys: 13.2 s, total: 16.9 s
Wall tim

In [21]:
kernel = kernel_channels[0]

print(kernel.shape)
print(type(kernel))

(524288, 524288)
<class 'tensorflow.python.framework.sparse_tensor.SparseTensor'>


In [25]:
col_sum = tf.sparse.reduce_sum(kernel, axis=1, output_is_sparse=False)

print(col_sum.shape)
print(type(col_sum))

(524288,)
<class 'tensorflow.python.framework.ops.EagerTensor'>


In [28]:
normalized_kernel = kernel / tf.expand_dims(col_sum, axis=0)

print(normalized_kernel.shape)
print(type(normalized_kernel))

(524288, 524288)
<class 'tensorflow.python.framework.sparse_tensor.SparseTensor'>


### real space smoothing by sparse-dense matmul

In [77]:
# maximum size allowed by tf.sparse.sparse_dense_matmul
op_size = len(kernel_channels[0].indices) * map_channels_batch[0].shape[1]
print(op_size < 2**31, op_size, "<", 2**31)

True 1908408320 < 2147483648


In [78]:
# %%time
# with tf.device("gpu"):
#     map_batch_conv = []
#     for i in range(n_channels):
#         m_conv = tf.sparse.sparse_dense_matmul(kernel_channels[i], map_channels_batch[i])
#         # m_conv = split_sparse_dense_matmul(kernel_channels[i], map_channels_batch[i], 2)

#         map_batch_conv.append(m_conv)
#     map_batch_conv = tf.stack(map_batch_conv)

In [115]:
%%time
map_batch_conv = []
for i in range(n_channels):
    # m_conv = tf.sparse.sparse_dense_matmul(kernel_channels[i], map_channels_batch[i])
    m_conv = split_sparse_dense_matmul(kernel_channels[i], map_channels_batch[i], 2)

    map_batch_conv.append(m_conv)
map_batch_conv = tf.stack(map_batch_conv)

CPU times: user 7.05 ms, sys: 10.5 ms, total: 17.6 ms
Wall time: 214 ms
